# Road Distress Instance Segmentation: YOLOv8-seg on Kaggle

Classes: `Alligator Distress`, `Longitudinal Distress`, `Pothole`, `Transverse Distress`

**Before running:**
1. Settings → Accelerator → **GPU T4 x2** (or P100). Settings → Internet → **On**.
2. Add-ons → Secrets → add `ROBOFLOW_API_KEY` (Roboflow → Settings → API Keys → Private key). Never paste the key into a cell.
3. `VERSION = 3` is the Roboflow version **"troy eto"** (Distress class names, Fit-1024, train-only 3x augmentation, 70/20/10 split).
4. Run all cells in order. Use **Save Version → Save & Run All (Commit)** so training survives closing the browser.

In [ ]:
!pip install -q -U ultralytics roboflow imagehash
import ultralytics, torch
ultralytics.checks()
print("GPUs:", torch.cuda.device_count())

## 1. Configuration

In [ ]:
from pathlib import Path

WORKSPACE = "gab-pkfzz"
PROJECT   = "my-first-project-oy4hj-lrohv"
VERSION   = 3                      # "troy eto": created after the class rename

EXPECTED_NAMES = ["Alligator Distress", "Longitudinal Distress", "Pothole", "Transverse Distress"]

MODEL   = "yolov8l-seg.pt"         # yolov8m-seg.pt if you run out of memory/time
IMGSZ   = 1024                     # cracks are thin: do NOT go below 1024
EPOCHS  = 100                      # v3 already holds 3 augmented copies of every train image
BATCH   = 16                       # T4 x2 -> 16 (8 per GPU). OOM? use 12 or 8. P100 -> 8
DEVICE  = "0,1" if torch.cuda.device_count() > 1 else "0"

NULL_FRACTION = 0.15               # keep at most 15% background (no-label) images in train
DUP_HAMMING   = 4                  # pHash distance <= this counts as the same photo

WORK = Path("/kaggle/working")
DATA = WORK / "data"
QUAR = WORK / "quarantine"         # images removed by the cleaning step (kept, not deleted)
RUN_NAME = "distress_v3_l1024"

## 2. Download the dataset from Roboflow

In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8", location=str(DATA), overwrite=True)
print(ds.location)

## 3. Audit and clean

Checks every label file, then:
* fails fast if the class names are not the new `... Distress` names,
* drops degenerate polygons (< 3 points / out of range coordinates),
* removes **val/test images that are near-duplicates of a train image** (leakage inflates scores and hides real performance),
* down-samples background (null) images in train to `NULL_FRACTION`. The Roboflow project has ~1,700 `uncracked_*` texture tiles (~41%), which drowns the distress classes.

In [ ]:
import yaml, random, shutil, collections
import numpy as np
from PIL import Image
import imagehash

random.seed(42)
cfg = yaml.safe_load(open(DATA / "data.yaml"))
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][i] for i in sorted(cfg["names"])]
print("names:", names)
assert names == EXPECTED_NAMES, f"Class names are {names}. Re-export a Roboflow version created AFTER the rename."

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
SPLITS = {"train": "train", "val": "valid", "test": "test"}

def images(split):
    return sorted(p for p in (DATA / split / "images").iterdir() if p.suffix.lower() in IMG_EXT)

def label_of(img):
    return img.parent.parent / "labels" / (img.stem + ".txt")

def quarantine(img, reason):
    d = QUAR / reason; (d / "images").mkdir(parents=True, exist_ok=True); (d / "labels").mkdir(parents=True, exist_ok=True)
    shutil.move(str(img), d / "images" / img.name)
    if label_of(img).exists():
        shutil.move(str(label_of(img)), d / "labels" / label_of(img).name)

def audit(title):
    print(f"\n=== {title} ===")
    for s in SPLITS.values():
        imgs = images(s); inst = collections.Counter(); null = 0
        for im in imgs:
            lines = [l for l in open(label_of(im)).read().splitlines() if l.strip()] if label_of(im).exists() else []
            null += not lines
            inst.update(names[int(l.split()[0])] for l in lines)
        print(f"{s:6s} images={len(imgs):5d} null={null:5d} ({null/max(len(imgs),1):.0%})  " +
              "  ".join(f"{n}={inst[n]}" for n in names))

audit("before cleaning")

# 3a. degenerate polygons
bad = 0
for s in SPLITS.values():
    for im in images(s):
        lp = label_of(im)
        if not lp.exists():
            continue
        keep = []
        for l in open(lp).read().splitlines():
            v = l.split()
            if not v:
                continue
            xy = np.array(v[1:], dtype=float)
            if int(v[0]) < len(names) and len(xy) >= 6 and len(xy) % 2 == 0 and (xy >= 0).all() and (xy <= 1).all():
                keep.append(l)
            else:
                bad += 1
        open(lp, "w").write("\n".join(keep) + ("\n" if keep else ""))
print("degenerate polygons removed:", bad)

In [ ]:
# 3b. near-duplicate leakage between train and val/test (perceptual hash)
def phash_bits(p):
    with Image.open(p) as im:
        return imagehash.phash(im.convert("RGB")).hash.flatten()

train_imgs = images("train")
H_train = np.stack([phash_bits(p) for p in train_imgs])
leaks = 0
for s in ("valid", "test"):
    for im in images(s):
        h = phash_bits(im)
        if (H_train != h).sum(1).min() <= DUP_HAMMING:
            quarantine(im, f"leak_{s}"); leaks += 1
print("val/test images removed as near-duplicates of train:", leaks)

# 3c. down-sample background images in train
nulls = [im for im in images("train") if not label_of(im).exists() or not open(label_of(im)).read().strip()]
labeled = len(images("train")) - len(nulls)
max_null = int(NULL_FRACTION * labeled / (1 - NULL_FRACTION))
random.shuffle(nulls)
for im in nulls[max_null:]:
    quarantine(im, "extra_null_train")
print(f"background images kept in train: {min(len(nulls), max_null)} / {len(nulls)}")

for p in DATA.rglob("*.cache"):
    p.unlink()

audit("after cleaning")

DATA_YAML = WORK / "distress.yaml"
yaml.safe_dump({"path": str(DATA), "train": "train/images", "val": "valid/images", "test": "test/images",
                "names": {i: n for i, n in enumerate(names)}}, open(DATA_YAML, "w"), sort_keys=False)
print(open(DATA_YAML).read())

## 4. Visual spot-check (look at this before spending 6 hours of GPU)

In [ ]:
import cv2, matplotlib.pyplot as plt
COLORS = [(230, 25, 75), (0, 200, 200), (210, 245, 60), (145, 30, 180)]
sample = random.sample([p for p in images("train") if label_of(p).exists() and open(label_of(p)).read().strip()], 12)
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for ax, p in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h, w = img.shape[:2]
    for l in open(label_of(p)).read().splitlines():
        v = l.split(); c = int(v[0])
        pts = (np.array(v[1:], float).reshape(-1, 2) * [w, h]).astype(np.int32)
        cv2.polylines(img, [pts], True, COLORS[c], max(2, w // 300))
    ax.imshow(img); ax.set_title(p.name[:40], fontsize=8); ax.axis("off")
plt.figtext(0.5, 0.01, "  ".join(f"{n}={c}" for n, c in zip(names, ["red", "cyan", "lime", "purple"])), ha="center", fontsize=14)
plt.show()

## 5. Train

Written to a script and launched with `!python` because multi-GPU (DDP) training is unreliable from inside a notebook kernel.

Why these settings:
* `imgsz=1024`: cracks are a few pixels wide and vanish at 640.
* `degrees=5`, no `flipud`, no 90° rotation: a big rotation turns a **Longitudinal** crack into a **Transverse** one, so the label would be wrong.
* `copy_paste=0.3`: pastes potholes and cracks onto other roads, which works well for segmentation.
* `cls=0.5` (default): the old run used `cls=3`, which over-weights the class loss and destabilises training.
* `cos_lr`, `close_mosaic=15`, `patience=30`: smooth LR decay and the last 15 epochs on real-looking images.
* v3 already has light offline augmentation (flip, crop, ±5° rotation, brightness, blur, noise) on **train only**, so val/test stay clean. The online augmentation here adds what Roboflow can't do on the free plan (mosaic, copy-paste, scale) and is kept moderate.

In [ ]:
%%writefile /kaggle/working/train.py
import sys, json
from ultralytics import YOLO

cfg = json.loads(sys.argv[1])
model = YOLO(cfg["model"])
model.train(
    data=cfg["data"], epochs=cfg["epochs"], imgsz=cfg["imgsz"], batch=cfg["batch"], device=cfg["device"],
    project=cfg["project"], name=cfg["name"], exist_ok=True,
    # optimisation
    optimizer="SGD", lr0=0.01, lrf=0.01, momentum=0.937, weight_decay=5e-4,
    warmup_epochs=3, cos_lr=True, patience=30, amp=True, seed=42,
    # loss weights (defaults)
    box=7.5, cls=0.5, dfl=1.5,
    # segmentation
    overlap_mask=True, mask_ratio=4,
    # augmentation (orientation-safe for longitudinal vs transverse)
    mosaic=1.0, close_mosaic=15, mixup=0.1, copy_paste=0.3,
    degrees=5.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.0, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4,
    # housekeeping
    workers=4, cache=False, plots=True, save_period=10, val=True,
)

In [ ]:
import json, subprocess
args = json.dumps({"model": MODEL, "data": str(DATA_YAML), "epochs": EPOCHS, "imgsz": IMGSZ,
                   "batch": BATCH, "device": DEVICE, "name": RUN_NAME,
                   "project": str(WORK / "runs")})
!python /kaggle/working/train.py '{args}'

**If the session dies mid-training:** start a new session, re-run sections 1 to 3, then run
`YOLO('/kaggle/working/runs/<RUN_NAME>/weights/last.pt').train(resume=True)`. To keep `last.pt` between sessions, save the notebook output as a Kaggle dataset.

## 6. Evaluate on the held-out TEST split

In [ ]:
from ultralytics import YOLO
import pandas as pd

RUN = WORK / "runs" / RUN_NAME
best = YOLO(RUN / "weights" / "best.pt")
r = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=8, conf=0.001, iou=0.6,
             plots=True, project=str(WORK / "runs"), name=RUN_NAME + "_test", exist_ok=True)

def f1(p, rcl):
    return 2 * p * rcl / (p + rcl + 1e-9)

rows = []
for i, c in enumerate(r.box.ap_class_index):   # i = row in the metric arrays, c = class id
    bp, br, bap50, bap = r.box.class_result(i)
    mp, mr, map50, mapx = r.seg.class_result(i)
    rows.append({"class": names[c], "box_P": bp, "box_R": br, "box_F1": f1(bp, br), "box_mAP50": bap50, "box_mAP50-95": bap,
                 "mask_P": mp, "mask_R": mr, "mask_F1": f1(mp, mr), "mask_mAP50": map50, "mask_mAP50-95": mapx})
df = pd.DataFrame(rows).set_index("class")
df.loc["ALL"] = df.mean()
pd.options.display.float_format = "{:.3f}".format
display(df)
df.to_csv(RUN / "test_metrics.csv")

### Image-level classification accuracy (separate metric, label it as such in the thesis)

For each test image and each class: does the image contain that distress (ground truth) and did the model predict it (at the confidence that maximised F1 on **val**)? This is the "accuracy" most readers expect. It is **not** mAP, so report both and say which is which.

In [ ]:
import json
# pick the confidence threshold on VAL (not test) to avoid tuning on the test set
val_r = best.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=8, conf=0.001, iou=0.6, plots=False,
                 project=str(WORK / "runs"), name=RUN_NAME + "_valthr", exist_ok=True)
f1_curve = val_r.seg.f1_curve.mean(0)              # mean F1 over classes vs confidence
CONF = float(np.linspace(0, 1, len(f1_curve))[f1_curve.argmax()])
print(f"best-F1 confidence on val: {CONF:.3f}")

tp = fp = fn = tn = 0; exact = 0; per = collections.defaultdict(lambda: [0, 0])
test_imgs = images("test")
for res in best.predict([str(p) for p in test_imgs], imgsz=IMGSZ, conf=CONF, stream=True, verbose=False):
    p = Path(res.path)
    gt = {int(l.split()[0]) for l in open(label_of(p)).read().splitlines() if l.strip()} if label_of(p).exists() else set()
    pr = set(res.boxes.cls.int().tolist()) if res.boxes is not None else set()
    exact += gt == pr
    for c in range(len(names)):
        g, q = c in gt, c in pr
        per[names[c]][0] += g == q; per[names[c]][1] += 1
        tp += g and q; fp += q and not g; fn += g and not q; tn += (not g) and (not q)
n = len(test_imgs)
print(f"Per-class presence accuracy (all classes): {(tp+tn)/(tp+tn+fp+fn):.3f}")
print(f"Exact-match accuracy (all classes right in an image): {exact/n:.3f}")
for c, (ok, tot) in per.items():
    print(f"  {c:22s} {ok/tot:.3f}")
json.dump({"conf": CONF}, open(RUN / "deploy_conf.json", "w"))

## 7. (Optional) Compare with the old model on the same test set

In [ ]:
# Upload the old backend/best.pt as a Kaggle dataset, attach it, and set the path.
OLD = Path("/kaggle/input/old-distress-model/best.pt")
if OLD.exists():
    old = YOLO(OLD)   # same class order, old names; metrics are index-based so the comparison is valid
    o = old.val(data=str(DATA_YAML), split="test", imgsz=1024, batch=8, conf=0.001, iou=0.6, plots=False,
                project=str(WORK / "runs"), name="old_on_v3_test", exist_ok=True)
    print(f"OLD  mask mAP50={o.seg.map50:.3f}  box mAP50={o.box.map50:.3f}")
    print(f"NEW  mask mAP50={r.seg.map50:.3f}  box mAP50={r.box.map50:.3f}")

## 8. Package the results

In [ ]:
shutil.copy(RUN / "weights" / "best.pt", WORK / "distress_best.pt")
shutil.make_archive(str(WORK / RUN_NAME), "zip", RUN)
print("Download from the Output tab:  distress_best.pt  and ", RUN_NAME + ".zip")